Import Required Libraries

In [ ]:
import re
import string
import emoji
import nltk
from collections import Counter

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize


Task 2 : Build Advanced Preprocessing Function

In [ ]:
STOP_WORDS = set(stopwords.words('english'))

MEANINGFUL_SHORT_WORDS = {'no', 'not', 'nor', 'but', 'yet'}

lemmatizer = WordNetLemmatizer()

print('Configuration loaded.')

Configuration loaded.


In [ ]:
#  HELPER FUNCTIONS
def remove_urls(text):

    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    return text

def remove_emojis(text):

    return emoji.replace_emoji(text, replace='')

def remove_numbers(text):
    return re.sub(r'\b\d+\b', '', text)


def handle_repeated_characters(text):
    result = []
    i = 0
    while i < len(text):
        char = text[i]
        run_len = 1
        # Count how many times this character repeats consecutively
        while i + run_len < len(text) and text[i + run_len] == char:
            run_len += 1
        if run_len >= 3:
            result.append(char)
        else:
            result.append(char * run_len)
        i += run_len
    return ''.join(result)

def remove_punctuation_and_special_chars(text):
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)


def convert_to_lowercase(text):
    return text.lower()


def remove_extra_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()


def filter_tokens(tokens):
    filtered = []
    for token in tokens:
        token = token.strip()
        if not token:
            continue
        if token in MEANINGFUL_SHORT_WORDS:
            filtered.append(token)
            continue
        if len(token) <= 2:
            continue
        filtered.append(token)
    return filtered


def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(token) for token in tokens]

print('All helper functions defined.')

All helper functions defined.


In [ ]:
#  MAIN PREPROCESSING FUNCTION
def preprocess_text(text):
    # Step 1: Input validation
    if text is None:
        return {'tokens': [], 'clean_sentence': ''}
    if not isinstance(text, str):
        text = str(text)
    if not text.strip():
        return {'tokens': [], 'clean_sentence': ''}

    text = remove_emojis(text)                      # Step 2
    text = remove_urls(text)                         # Step 3
    text = handle_repeated_characters(text)          # Step 4
    text = convert_to_lowercase(text)                # Step 5
    text = remove_numbers(text)                      # Step 6
    text = remove_punctuation_and_special_chars(text)# Step 7
    text = remove_extra_spaces(text)                 # Step 8

    if not text:
        return {'tokens': [], 'clean_sentence': ''}

    tokens = text.split()                            # Step 9
    tokens = filter_tokens(tokens)                   # Step 10
    tokens = lemmatize_tokens(tokens)                # Step 11

    return {'tokens': tokens, 'clean_sentence': ' '.join(tokens)}

checks = [
    ("I absolutely looooved this product","'loved' must survive (4 o's → collapsed)"),
    ("Get 100% FREE access now!!!","'free' and 'access' must survive (2-char runs kept)"),
    ("Nooooo this is baaad!!!","'no' and 'bad' expected"),
    ("I am not happy with this","'not' and 'happy' must survive"),
    ("Win $$$ now!!! Limited offer!!!","'offer' must survive (2 f's kept)"),
    ("Call me at 9876543210","phone number removed; 'call' survives (2 l's kept)"),
]

print('─' * 62)
print(' SANITY CHECKS')
print('─' * 62)
for raw, note in checks:
    r = preprocess_text(raw)
    print(f'  Input : {raw}')
    print(f'  Note  : {note}')
    print(f'  Output: {r["clean_sentence"]}')
    print()
print('All sanity checks passed!')

──────────────────────────────────────────────────────────────
 SANITY CHECKS
──────────────────────────────────────────────────────────────
  Input : I absolutely looooved this product
  Note  : 'loved' must survive (4 o's → collapsed)
  Output: absolutely loved this product

  Input : Get 100% FREE access now!!!
  Note  : 'free' and 'access' must survive (2-char runs kept)
  Output: get free access now

  Input : Nooooo this is baaad!!!
  Note  : 'no' and 'bad' expected
  Output: no this bad

  Input : I am not happy with this
  Note  : 'not' and 'happy' must survive
  Output: not happy with this

  Input : Win $$$ now!!! Limited offer!!!
  Note  : 'offer' must survive (2 f's kept)
  Output: win now limited offer

  Input : Call me at 9876543210
  Note  : phone number removed; 'call' survives (2 l's kept)
  Output: call

All sanity checks passed!


Task 3 : Stress Testing

In [ ]:
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

def run_stress_test(sentences):
    print('=' * 70)
    print('         STRESS TEST RESULTS — NLP PREPROCESSING ENGINE')
    print('=' * 70)
    results = []
    for i, sentence in enumerate(sentences, 1):
        result = preprocess_text(sentence)
        results.append(result)
        print(f'\n[{i}] Original Text   : {sentence}')
        print(f'Cleaned Tokens  : {result["tokens"]}')
        print(f'Cleaned Sentence: {result["clean_sentence"] or "(empty after cleaning)"}')
        print('-' * 70)
    return results

stress_results = run_stress_test(sample_inputs)

         STRESS TEST RESULTS — NLP PREPROCESSING ENGINE

[1] Original Text   : Get 100% FREE access now!!!
Cleaned Tokens  : ['get', 'free', 'access', 'now']
Cleaned Sentence: get free access now
----------------------------------------------------------------------

[2] Original Text   : I absolutely looooved this product 😍😍
Cleaned Tokens  : ['absolutely', 'loved', 'this', 'product']
Cleaned Sentence: absolutely loved this product
----------------------------------------------------------------------

[3] Original Text   : Worst service ever... 0/10
Cleaned Tokens  : ['worst', 'service', 'ever']
Cleaned Sentence: worst service ever
----------------------------------------------------------------------

[4] Original Text   : Call me at 9876543210
Cleaned Tokens  : ['call']
Cleaned Sentence: call
----------------------------------------------------------------------

[5] Original Text   : This is THE best course!!!
Cleaned Tokens  : ['this', 'the', 'best', 'course']
Cleaned Sentence: t

Task 4 : Token Analytics

In [ ]:
def compute_token_analytics(original_sentences, preprocessed_results):
    print('=' * 70)
    print('TOKEN ANALYTICS REPORT')
    print('=' * 70)
    print(f'  {"#":<4} {"Total":>7} {"Unique":>7} {"Avg Len":>9}  Cleaned sentence')
    print('  ' + '-' * 66)

    analytics = []
    for i, (original, result) in enumerate(zip(original_sentences, preprocessed_results), 1):
        tokens  = result['tokens']
        total   = len(tokens)
        unique  = len(set(tokens))
        avg_len = round(sum(len(t) for t in tokens) / total, 2) if tokens else 0
        removed = len(original.split()) - total

        analytics.append({
            'index': i, 'original': original,
            'total': total, 'unique': unique,
            'avg_len': avg_len, 'removed': removed
        })

        preview = result['clean_sentence'][:42] + ('...' if len(result['clean_sentence']) > 42 else '')
        print(f'{i:<4} {total:>7} {unique:>7} {avg_len:>9}  {preview or "(empty)"}')

    print('=' * 70)

    noisy = max(analytics, key=lambda x: x['removed'])
    rich  = max(analytics, key=lambda x: x['total'])

    print(f'\nMost noisy sentence(#{noisy["index"]}): "{noisy["original"]}"')
    print(f'→ {noisy["removed"]} token(s) removed during cleaning.')
    print(f'\nMost content retained  (#{rich["index"]}): "{rich["original"]}"')
    print(f' → {rich["total"]} clean token(s) kept.')

    return analytics


analytics_data = compute_token_analytics(sample_inputs, stress_results)

TOKEN ANALYTICS REPORT
  #      Total  Unique   Avg Len  Cleaned sentence
  ------------------------------------------------------------------
1          4       4       4.0  get free access now
2          4       4       6.5  absolutely loved this product
3          3       3      5.33  worst service ever
4          1       1       4.0  call
5          4       4      4.25  this the best course
6          2       2       4.0  visit now
7          3       3       3.0  no this bad
8          1       1       3.0  got
9          4       4       4.5  win now limited offer
10         4       4       4.0  not happy with this

Most noisy sentence(#8): "OK OK OK I got it"
→ 5 token(s) removed during cleaning.

Most content retained  (#1): "Get 100% FREE access now!!!"
 → 4 clean token(s) kept.


Task 5 : Frequency Analysis

In [ ]:
def frequency_analysis(preprocessed_results):
    all_tokens = []
    for result in preprocessed_results:
        all_tokens.extend(result['tokens'])

    if not all_tokens:
        print('No tokens found.')
        return Counter()

    counts = Counter(all_tokens)

    print('=' * 50)
    print('FREQUENCY ANALYSIS REPORT')
    print('=' * 50)
    print(f'Total tokens  : {len(all_tokens)}')
    print(f'Unique tokens : {len(counts)}')

    print('\nTop 10 most frequent words:')
    print(f'{"Rank":<6} {"Word":<15} {"Count":>5}')
    print('  ' + '-' * 28)
    for rank, (word, count) in enumerate(counts.most_common(10), 1):
        print(f'  {rank:<6} {word:<15} {count:>5}')

    print('\nTop 5 least frequent words:')
    print(f'{"Rank":<6} {"Word":<15} {"Count":>5}')
    print('  ' + '-' * 28)
    for rank, (word, count) in enumerate(counts.most_common()[:-6:-1], 1):
        print(f'  {rank:<6} {word:<15} {count:>5}')

    print('=' * 50)
    return counts


token_frequency = frequency_analysis(stress_results)

FREQUENCY ANALYSIS REPORT
Total tokens  : 30
Unique tokens : 25

Top 10 most frequent words:
Rank   Word            Count
  ----------------------------
  1      this                4
  2      now                 3
  3      get                 1
  4      free                1
  5      access              1
  6      absolutely          1
  7      loved               1
  8      product             1
  9      worst               1
  10     service             1

Top 5 least frequent words:
Rank   Word            Count
  ----------------------------
  1      with                1
  2      happy               1
  3      not                 1
  4      offer               1
  5      limited             1


Task 6 : Build Full Pipeline

In [ ]:
def full_pipeline(text_list):
    if not text_list or not isinstance(text_list, list):
        print('Input must be a non-empty list of strings.')
        return {'tokens': [], 'clean_sentences': []}

    all_tokens, clean_sentences = [], []

    for text in text_list:
        result = preprocess_text(text)
        all_tokens.extend(result['tokens'])
        clean_sentences.append(result['clean_sentence'])

    return {'tokens': all_tokens, 'clean_sentences': clean_sentences}


pipeline_output = full_pipeline(sample_inputs)

print('=' * 60)
print('           FULL PIPELINE OUTPUT')
print('=' * 60)
print(f'\nAll Tokens ({len(pipeline_output["tokens"])} total):')
print(pipeline_output['tokens'])
print(f'\nClean Sentences ({len(pipeline_output["clean_sentences"])} total):')
for i, s in enumerate(pipeline_output['clean_sentences'], 1):
    print(f'  [{i}] {s if s else "(empty after cleaning)"}')
print('\nFull pipeline executed successfully!')

           FULL PIPELINE OUTPUT

All Tokens (30 total):
['get', 'free', 'access', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this']

Clean Sentences (10 total):
  [1] get free access now
  [2] absolutely loved this product
  [3] worst service ever
  [4] call
  [5] this the best course
  [6] visit now
  [7] no this bad
  [8] got
  [9] win now limited offer
  [10] not happy with this

Full pipeline executed successfully!


Task 7 : Error Handling

In [ ]:
edge_cases = [
    '',              # Empty string
    '   ',           # Whitespace only
    '😍🔥💯😂🎉',    # Only emojis
    '1234567890',    # Only numbers
    None,            # None input
    12345,           # Integer (non-string)
    '!!!! #### $$$', # Only special characters
]

print('=' * 60)
print('ERROR HANDLING — EDGE CASE RESULTS')
print('=' * 60)

for i, case in enumerate(edge_cases, 1):
    try:
        result = preprocess_text(case)
        status = 'Handled'
    except Exception as e:
        result = {'tokens': [], 'clean_sentence': f'ERROR: {e}'}
        status = 'Error'

    print(f'\n[{i}] Input   : {repr(case)[:50]}')
    print(f'Status  : {status}')
    print(f'Tokens  : {result["tokens"]}')
    print(f'Output  : "{result["clean_sentence"]}"')

print('\n' + '=' * 60)
print('All edge cases handled gracefully — no crashes!')

ERROR HANDLING — EDGE CASE RESULTS

[1] Input   : ''
Status  : Handled
Tokens  : []
Output  : ""

[2] Input   : '   '
Status  : Handled
Tokens  : []
Output  : ""

[3] Input   : '😍🔥💯😂🎉'
Status  : Handled
Tokens  : []
Output  : ""

[4] Input   : '1234567890'
Status  : Handled
Tokens  : []
Output  : ""

[5] Input   : None
Status  : Handled
Tokens  : []
Output  : ""

[6] Input   : 12345
Status  : Handled
Tokens  : []
Output  : ""

[7] Input   : '!!!! #### $$$'
Status  : Handled
Tokens  : []
Output  : ""

All edge cases handled gracefully — no crashes!
